# Fraud Detection – Model Training

This notebook implements the model training pipeline based on the preprocessed 
data prepared during the preprocessing stage. The goal is to train baseline and 
advanced classifiers, handle class imbalance, and return fitted models ready for 
evaluation.

## 1. Import Libraries


In [12]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

# Baseline classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Advanced classifiers
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Evaluation
from sklearn.metrics import roc_auc_score

# Reproducibility seed
SEED = 42
np.random.seed(SEED)
print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Load Processed Data

We load the preprocessed train/test splits saved from notebook `02_preprocessing.ipynb`.


In [13]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test  = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test  = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train fraud rate: {:.2%}".format(y_train.mean()))
print("y_test  fraud rate: {:.2%}".format(y_test.mean()))

X_train: (472432, 422)
X_test : (118108, 422)
y_train fraud rate: 3.50%
y_test  fraud rate: 3.50%


## 3. Handle Class Imbalance

The dataset is heavily imbalanced (fraud transactions are a small minority).  
We compute `scale_pos_weight` — the ratio of negatives to positives — which XGBoost uses to upweight the minority class during training.  
For other models we pass `class_weight='balanced'` which applies the same principle automatically.


In [14]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f"Non-fraud (0): {neg:,}")
print(f"Fraud     (1): {pos:,}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

Non-fraud (0): 455,902
Fraud     (1): 16,530
scale_pos_weight: 27.58


## 4. Training Configuration

All model hyperparameters are defined here in one place to keep training reproducible and easy to tune.


In [15]:
# ── Baseline Models ──────────────────────────────────────────────────────────
baseline_models = {
    "LogisticRegression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=SEED
    ),
    "DecisionTree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=6,
        random_state=SEED
    ),
}

In [26]:
# ── Advanced Models ───────────────────────────────────────────────────────────
advanced_models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        class_weight="balanced",
        random_state=SEED
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,   # handles imbalance
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=SEED
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        random_state=SEED,
        scale_pos_weight=scale_pos_weight,
        verbose=-1
    ),
}

print("Baseline models :", list(baseline_models.keys()))
print("Advanced models :", list(advanced_models.keys()))

Baseline models : ['LogisticRegression', 'DecisionTree']
Advanced models : ['RandomForest', 'XGBoost', 'LightGBM']


## 5. Train Baseline Classifiers

Logistic Regression and Decision Tree serve as benchmarks.  
Their performance defines the minimum bar the advanced models must beat.


In [23]:
baseline_results = {}
fitted_baseline  = {}

for name, model in baseline_models.items():
    print(f"Training {name} ...", end=" ")
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    baseline_results[name] = auc
    fitted_baseline[name]  = model
    print(f"ROC-AUC = {auc:.4f}")

print("\nBaseline training complete.")

Training LogisticRegression ... ROC-AUC = 0.8617
Training DecisionTree ... ROC-AUC = 0.8410

Baseline training complete.


## 6. Train Advanced Classifiers

XGBoost and LightGBM are gradient-boosted tree ensembles that consistently 
outperform simpler models on tabular fraud data.  
Random Forest is included as a non-boosting reference, though it shows lower 
performance compared to the boosting models on this dataset.

In [27]:
advanced_results = {}
fitted_advanced  = {}

for name, model in advanced_models.items():
    print(f"Training {name} ...", end=" ")
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    advanced_results[name] = auc
    fitted_advanced[name]  = model
    print(f"ROC-AUC = {auc:.4f}")

print("\nAdvanced training complete.")

Training RandomForest ... ROC-AUC = 0.8873
Training XGBoost ... ROC-AUC = 0.9325
Training LightGBM ... ROC-AUC = 0.9402

Advanced training complete.


## 7. Save Fitted Models

All fitted models are persisted to `models/` so the evaluation notebook can load them directly without retraining.


In [29]:
os.makedirs('../models', exist_ok=True)

all_fitted = {**fitted_baseline, **fitted_advanced}

for name, model in all_fitted.items():
    filepath = f'../models/{name.lower()}.pkl'
    joblib.dump(model, filepath)
    print(f'Saved: {filepath}')

print('\nAll models saved — ready for notebook 05_evaluation.')

Saved: ../models/logisticregression.pkl
Saved: ../models/decisiontree.pkl
Saved: ../models/randomforest.pkl
Saved: ../models/xgboost.pkl
Saved: ../models/lightgbm.pkl

All models saved — ready for notebook 05_evaluation.


## Final Summary

| Task | How it was handled |
|---|---|
| **Baseline classifiers** | Logistic Regression, Decision Tree with `class_weight='balanced'` |
| **Advanced classifiers** | Random Forest, XGBoost, LightGBM |
| **Class imbalance** | `scale_pos_weight` for XGBoost & LightGBM; `class_weight='balanced'` for sklearn models |
| **Reproducibility** | `random_state=42` set on every model |
| **Evaluation metric** | ROC-AUC printed per model during training (not full evaluation) |
| **Saved artifacts** | One `.pkl` per model in `models/` |

The fitted models are ready for the next stage: **05_evaluation**.
